# GIF Creator

## Imports and files

In [1]:
import os
import pickle
import scipy.io as sio
import itertools
from datetime import datetime, timedelta
import re

import numpy as np
import pandas as pd
import geopandas as gpd

import contextily as ctx
from shapely.geometry import Polygon, Point

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from rasterio.transform import from_origin
from collections import namedtuple

from swmm_api.input_file import read_inp_file, SwmmInput, section_labels as sections
from swmm_api import read_out_file, swmm5_run

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pyproj import Transformer

import h5py

import matplotlib.pyplot as plt
import imageio.v2 as imageio
import pandas as pd
import geopandas as gpd

# All events SWMM results

# Function

In [2]:
def process_wrf_data(rain_array_3d, raanana_basins_gdf, metadata, time_vector, wrf_basin_legend_gdf, gif_path='rainfall_evolution.gif', time_start=100, time_end=300):

    time_steps = rain_array_3d.shape[0]

    time_diff = time_vector[1:] - time_vector[:-1]
    is_10_min_interval = (time_diff == pd.Timedelta(minutes=10)).all()

    if not is_10_min_interval:
        raise ValueError("Time differences are not exactly 10 minutes. Please check the time vector.")

    wrf_gdf_list = []

    for time in range(time_start, min(time_end, time_steps)):
        rain_array_2d = rain_array_3d[time, :, :]
        wrf_gdf = wrf_array_to_gdf(rain_array_2d, **metadata)

        wrf_intersecting_basins = gpd.sjoin(wrf_gdf, raanana_basins_gdf, how="inner", predicate="intersects")
        wrf_indices = wrf_intersecting_basins.index.unique()
        wrf_filtered_gdf = wrf_gdf.loc[wrf_indices]
        wrf_gdf_list.append(wrf_filtered_gdf['rain_value'])

    wrf_poly_grid_gdf = pd.concat(wrf_gdf_list, axis=1)
    wrf_poly_grid_gdf.columns = time_vector

    wrf_poly_grid_gdf['geometry'] = wrf_gdf['geometry']
    wrf_poly_grid_gdf = gpd.GeoDataFrame(wrf_poly_grid_gdf, crs='epsg:2039')
    wrf_poly_grid_gdf.index.name = 'wrf_cell_num'

    basin_rain_columns = ['Basin_name'] + list(wrf_poly_grid_gdf.columns[:])
    basins_rain_df = pd.DataFrame()
    for basin in range(1, len(raanana_basins_gdf) + 1):
        basin_df = wrf_basin_legend_gdf[wrf_basin_legend_gdf['Basin_name'] == basin][['Basin_name', 'pct', 'wrf_cell_num']]
        basin_wrf_cells_df = wrf_poly_grid_gdf.loc[basin_df['wrf_cell_num']]
        basin_wrf_cells_df = pd.merge(basin_df, basin_wrf_cells_df, on='wrf_cell_num')
        weighted_avgs = []
        for col in basin_rain_columns[1:-1]:
            weighted_avg = sum(basin_wrf_cells_df[col] * basin_wrf_cells_df['pct'])
            weighted_avgs.append(weighted_avg)
        new_row = [basin] + weighted_avgs
        df_basin = pd.DataFrame([new_row], columns=basin_rain_columns[:-1])
        basins_rain_df = pd.concat([basins_rain_df, df_basin])
    basins_rain_df = basins_rain_df.reset_index(drop=True)
    basins_rain_df.fillna(0, inplace=True)

    # Plot and save the final GIF
    images = []
    for time in range(time_start, min(time_end, time_steps)):
        rain_array_2d = rain_array_3d[time, :, :]
        plt.figure(figsize=(8, 6))
        im = plt.imshow(rain_array_2d, cmap='viridis', origin='lower', aspect='auto', vmin=0, vmax=150)  # Adjust vmin and vmax for color bar range
        plt.colorbar(im, label='Rainfall (mm)', extend='both')
        plt.title(f'Rainfall Array at Time Step {time} ({time_vector[time]})')
        plt.xlabel('X-coordinate')
        plt.ylabel('Y-coordinate')
        image_path = f'temp_plot_{time}.png'
        plt.savefig(image_path)
        images.append(imageio.imread(image_path))
        plt.close()

    imageio.mimsave(gif_path, images, duration=0.1)  # Adjust duration as needed

    return basins_rain_df

## Load and process data

In [3]:
# Raanana sub-basin shapefile
raanana_basins_shapfile = r'D:\Development\RESEARCH\Raanana\gis\GIS\28_subcatchments\raanana_28_subcatchments.shp'
raanana_basins_gdf = gpd.read_file(raanana_basins_shapfile)  # sub-basins poly
raanana_basins_gdf.drop(columns=['Shape_Area', 'Area_km2', 'Area_ha','Area_m2'], inplace = True)

# shoreline_path = r'D:\Development\RESEARCH\Raanana\gis\GIS\shoreline\iho.shp'
# shoreline_gdf = gpd.read_file(shoreline_path)

# Define the path to the .mat file
# mat_path = r'\\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\matfiles\future\WRFrain_PGW_V3_HPE10.mat'
mat_path = r'\\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\matfiles\historical\WRFrain_noPGW_HPE38.mat'
## Select time for the plot and to extract one general rain 2d array
time_i = 0
filename = os.path.basename(mat_path).split('.')[0]
# Read the .mat file
with h5py.File(mat_path, 'r') as file:
    if 'future' in mat_path:
        rain_rate_data = np.array(file['rainRatePGW'])
        rain_total_data = np.array(file['totalRainPGW'])
        
    elif 'historical' in mat_path:
        rain_rate_data = np.array(file['rainRate'])
        rain_total_data = np.array(file['totalRain'])
        
    # Extract data
    lat_data = np.array(file['lat'])
    lon_data = np.array(file['lon'])
    time_data = np.array(file['timesList']).flatten()
    time_vector = pd.to_datetime(time_data - 719529, unit='D').round('1min')
    
    # Print available keys
#     print("Keys: %s" % file.keys())

## clip a relevent domain from the array
lon_domain_strat, lon_domain_end = 50, 500 
lat_domain_strat, lat_domain_end = 50, 500 

rain_array_3d = rain_rate_data[:, lat_domain_strat:lat_domain_end, lon_domain_strat:lon_domain_end]
rain_array_2d = rain_array_3d[time_i, :, :]  # Access only one time slice

# Define the coordinate reference system transformation using pyproj.Transformer
transformer = Transformer.from_crs("epsg:4326", "epsg:2039", always_xy=True)
lon_transformed, lat_transformed = transformer.transform(lon_data, lat_data)

# Ensure the transformed coordinates are correctly reshaped
lon_transformed = (lon_transformed.reshape(lon_data.shape)).astype(int)
lat_transformed = (lat_transformed.reshape(lat_data.shape)).astype(int)

# Define the cell size and domain limits
xll = lon_transformed.min() + (lon_domain_strat*1000)
xhr = lon_transformed.min() + (lon_domain_end*1000)
yll = lat_transformed.min() + (lat_domain_strat*1000)
yhr = lat_transformed.min() + (lat_domain_end*1000)

cellsize = 1000

## Create te gif

# Initialize an empty list to store the images
images = []

# Define constants for color bar limits
colorbar_min = 0
colorbar_max = 150

# Create a custom colormap where values less than 1 are white and others use 'viridis'
cmap = plt.cm.viridis
cmap.set_under('white')

# Iterate over all time indices and plot where conditions are met
for time_index in range(len(time_vector)):
    rain_array_2d = rain_array_3d[time_index, :, :]

    # Replace rain values less than 1 with NaN (to display as white in the plot)
    rain_array_2d[rain_array_2d < 1] = np.nan

    # Plot the rainfall data
    plt.figure(figsize=(10, 8))
    plt.imshow(rain_array_2d, extent=[xll, xhr, yll, yhr], origin='lower', cmap=cmap, aspect='auto',
               vmin=colorbar_min, vmax=colorbar_max)
    
    # Plot Raanana basin boundaries
    raanana_basins_gdf.boundary.plot(ax=plt.gca(), color='red', linewidth=1)

    # Plot shoreline
    shoreline_gdf.boundary.plot(ax=plt.gca(), color='blue', linewidth=1)
    
    plt.colorbar(label='Rain Rate (mm/h)', ticks=np.linspace(colorbar_min, colorbar_max, 6), extend='both')
    plt.title(f"Rain Rate at {time_vector[time_index]}")
    plt.xlabel('Longitude (m)')
    plt.ylabel('Latitude (m)')
    plt.grid(True)

    # Save the plot as an image temporarily
    filename = f'rain_plot_{time_index}.png'
    plt.savefig(filename)
    plt.close()

    # Append the image file to the list
    images.append(imageio.imread(filename))

    # Remove the temporary image file
    os.remove(filename)

    
# Save gif section    
match = re.search(r'HPE\d+', mat_path)
if match:
    suffix = match.group(0)
else:
    raise ValueError("The suffix 'HPExx' not found in the .mat file path")

# Determine if the file is future or historical based on the path
if 'future' in mat_path:
    subdir = 'future'
elif 'historical' in mat_path:
    subdir = 'historical'
else:
    raise ValueError("The file path does not contain 'future' or 'historical'")
# Define the path where the GIF will be saved
save_path = rf'\\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\gifs\{subdir}\rainfall_animation_{suffix}.gif'
# Ensure the subdirectory exists
os.makedirs(os.path.dirname(save_path), exist_ok=True)

# Save the images as a GIF
imageio.mimsave(save_path, images, duration=0.1)



# Display a message once the GIF is created
print("GIF created successfully!")

C:\Users\raznu\AppData\Local\Temp\ipykernel_22412\1706342257.py:68: MatplotlibDeprecationWarning: You are modifying the state of a globally registered colormap. This has been deprecated since 3.3 and in 3.6, you will not be able to modify a registered colormap in-place. To remove this warning, you can make a copy of the colormap first. cmap = mpl.cm.get_cmap("viridis").copy()
  cmap.set_under('white')


GIF created successfully!
